# Imputation Technique 2: Constant Imputation

**Dataset:** `Loan_Default.csv`

**When to use:** When missing data is believed to be **Missing Not At Random (MNAR)** — i.e., the absence itself carries meaning. A sentinel value (like `0` or `-1`) signals to the model that this value was deliberately absent.

---

### Step 1: Setup — Data Loading & Prep

In [ ]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer

# Load data
df = pd.read_csv('../../data/raw/Loan_Default.csv')
df.drop(['ID', 'year'], axis=1, inplace=True)

categorical_features = df.select_dtypes(include=['object']).columns.tolist()
Ordinal_features = ['age']
Nominal_features = categorical_features.copy()
Nominal_features.remove('age')

enc = OrdinalEncoder()
df[Ordinal_features] = enc.fit_transform(df[Ordinal_features])

df_freq = df.copy()
for c in Nominal_features:
    df_freq[c + '_freq'] = df_freq[c].map(df_freq.groupby(c).size() / df_freq.shape[0])
    indexer = pd.factorize(df_freq[c], sort=True)[1]
    df_freq[c] = indexer.get_indexer(df_freq[c])
df_freq = df_freq.drop(Nominal_features, axis=1)

# Drop high-missing columns first
high_missing_cols = ['rate_of_interest', 'Interest_rate_spread', 'Upfront_charges', 'property_value', 'LTV', 'dtir1']
df_drop = df_freq.drop(high_missing_cols, axis=1)

print(f'Starting shape: {df_drop.shape}')

### Step 2: Define Columns to Impute

In [ ]:
Cols_to_be_imputed = [
    'term', 'income', 'age',
    'loan_limit_freq', 'approv_in_adv_freq', 'loan_purpose_freq',
    'Neg_ammortization_freq', 'submission_of_application_freq'
]

### Step 3: Apply Constant Imputation

`strategy='constant'` fills all NaNs with a fixed value (default is `0`).

In [ ]:
df_const = df_drop.copy()

Const_imputer = SimpleImputer(strategy='constant') # Defaults to 0
df_const[Cols_to_be_imputed] = Const_imputer.fit_transform(df_const[Cols_to_be_imputed])

df_const.head()

### Step 4: Verify Results

In [ ]:
missing_after = df_const.isna().sum()
print('Missing values after Constant Imputation:')
print(missing_after[missing_after > 0] if missing_after.sum() > 0 else 'None — all filled!')